# Food Agent Handoff Workflow

This notebook demonstrates the **handoff pattern** using the Microsoft Agent Framework. The handoff pattern allows one agent to transfer control to another specialized agent when the task requires specific expertise.

## What You'll Learn

- What is **Handoff** and when to use it
- How to create specialized sub-agents
- How to implement handoff between agents
- How to pass context during handoff

---

**Reference**: [Microsoft Learn - Handoff Pattern](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/handoff?pivots=programming-language-python)

## 1. Setup and Installations

First, let's install the required dependencies and set up the environment.

In [19]:
# Install required dependencies
!pip install python-dotenv

In [20]:
# Import and setup environment
import os
from dotenv import load_dotenv, find_dotenv
from agent_framework.openai import OpenAIChatClient

# Load environment variables
load_dotenv(find_dotenv())
import nest_asyncio
nest_asyncio.apply()
# Verify API keys are set
print(f"GROQ Endpoint: {os.environ.get('GROQ_ENDPOINT', 'Not set')}")
print(f"GROQ API Key: {'Set' if os.environ.get('GROQ_API_KEY') else 'Not set'}")

GROQ Endpoint: https://api.groq.com/openai/v1/
GROQ API Key: Set


In [39]:
# Create OPEN AI Chat Client for OpenRouter Models
# Using Groq as an alternative (faster, more reliable than Ollama for streaming)
openai_chat_client = OpenAIChatClient(
    base_url=os.environ.get("GROQ_ENDPOINT"),
    api_key=os.environ.get("GROQ_API_KEY"),
    model_id="openai/gpt-oss-120b"
)

In [40]:
# from agent_framework.openai import OpenAIChatClient
# # Create OPEN AI Chat Client for OpenRouter Models
# openai_chat_client = OpenAIChatClient(
#     base_url=os.environ.get("OPENROUTER_ENDPOINT"),
#     api_key=os.environ.get("OPENROUTER_API_KEY"),
#     model_id="nvidia/nemotron-3-nano-30b-a3b:free"
# )

---

## 2. Understanding the Handoff Pattern

### What is Handoff?

**Handoff** is a pattern where one agent transfers control to another agent. This is useful when:

- A general-purpose agent needs specialized knowledge
- Task complexity requires different expertise
- You want to delegate to an agent with specific tools
- Work needs to be split among specialized agents

### Key Concepts

- **Source Agent**: The agent that initiates the handoff
- **Target Agent**: The agent that receives the handoff
- **Handoff Tool**: A function that triggers the transfer
- **Context Transfer**: Passing relevant information to the target agent

## 3. Import Required Libraries

Let's import the necessary components from the Microsoft Agent Framework.

In [41]:
import asyncio
import os
import json
import requests
from random import randint
from typing import Annotated, Dict, Any, Optional

from agent_framework import tool, Agent
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential
from pydantic import Field

In [42]:
@tool
def process_refund(order_number: Annotated[str, "Order number to process refund for"]) -> str:
    """Simulated function to process a refund for a given order number."""
    return f"Refund processed successfully for order {order_number}."

@tool
def check_order_status(order_number: Annotated[str, "Order number to check status for"]) -> str:
    """Simulated function to check the status of a given order number."""
    return f"Order {order_number} is currently being processed and will ship in 2 business days."

@tool
def process_return(order_number: Annotated[str, "Order number to process return for"]) -> str:
    """Simulated function to process a return for a given order number."""
    return f"Return initiated successfully for order {order_number}. You will receive return instructions via email."

In [43]:
import asyncio
import json
import requests
from typing import Annotated, cast

from agent_framework import (
    Message,
    WorkflowEvent,
    WorkflowRunState,
    tool,
)
from agent_framework._types import AgentResponseUpdate
from agent_framework.orchestrations import HandoffAgentUserRequest, HandoffBuilder

---

## 4. Create Food-Related Tools

We'll create tools that our specialized agents will use. These tools interact with TheMealDB API.

In [44]:
# Helper function to clean meal data from API
def _clean_meal_data(meal: Dict[str, Any]) -> Dict[str, Any]:
    """
    Helper function to restructure the raw meal API response into a clean, 
    LLM-friendly format by combining ingredients and measures.
    """
    if not meal:
        return {}

    # Combine ingredients and measures into a single list
    ingredients = []
    for i in range(1, 21):
        ing = meal.get(f"strIngredient{i}")
        measure = meal.get(f"strMeasure{i}")
        if ing and ing.strip():
            ingredients.append(f"{measure.strip()} {ing.strip()}".strip())

    return {
        "id": meal.get("idMeal"),
        "name": meal.get("strMeal"),
        "category": meal.get("strCategory"),
        "area": meal.get("strArea"),
        "instructions": meal.get("strInstructions"),
        "ingredients": ingredients,
        "tags": meal.get("strTags"),
        "youtube_link": meal.get("strYoutube"),
        "image_url": meal.get("strMealThumb")
    }

In [45]:
# Tool: Get a random meal
@tool
def get_random_meal() -> str:
    """
    Retrieves a random meal recipe from the database. 
    Use this when the user wants a surprise suggestion or explicitly asks for a random recommendation.

    Returns:
        str: A JSON string containing the meal name, ingredients, and cooking instructions.
    """
    try:
        response = requests.get("https://www.themealdb.com/api/json/v1/1/random.php")
        response.raise_for_status()
        data = response.json()
        
        if not data.get("meals"):
            return json.dumps({"error": "No meal found."})
            
        meal = _clean_meal_data(data["meals"][0])
        return json.dumps(meal, indent=2)
        
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch random meal: {str(e)}"})

In [46]:
# Tool: Get meal by name
@tool
def get_meal_by_name(meal_name: Annotated[str, "The name of the meal to search for"]) -> str:
    """
    Search for a specific meal by its name.
    Use this when the user specifies a particular dish they want to make or learn about.

    Args:
        meal_name (str): The name of the meal to search for.

    Returns:
        str: A JSON string containing the meal details or search results.
    """
    try:
        response = requests.get(
            f"https://www.themealdb.com/api/json/v1/1/search.php?s={meal_name}"
        )
        response.raise_for_status()
        data = response.json()
        
        if not data.get("meals"):
            return json.dumps({"error": f"No meal found with name '{meal_name}'"})
        
        meals = [_clean_meal_data(meal) for meal in data["meals"]]
        return json.dumps(meals, indent=2)
        
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch meal: {str(e)}"})

In [47]:
# Tool: Get meals by category
@tool
def get_meals_by_category(category: Annotated[str, "The category name (e.g., Beef, Chicken, Dessert)"]) -> str:
    """
    Get a list of meals from a specific category (e.g., Beef, Chicken, Dessert).
    Use this when the user wants to explore meals from a specific category.

    Args:
        category (str): The category name (e.g., "Beef", "Chicken", "Dessert").

    Returns:
        str: A JSON string containing list of meals in that category.
    """
    try:
        response = requests.get(
            f"https://www.themealdb.com/api/json/v1/1/filter.php?c={category}"
        )
        response.raise_for_status()
        data = response.json()
        
        if not data.get("meals"):
            return json.dumps({"error": f"No meals found in category '{category}'"})
        
        meals = [{"id": m["idMeal"], "name": m["strMeal"], "thumbnail": m["strMealThumb"]} 
                 for m in data["meals"]]
        return json.dumps(meals[:10], indent=2)  # Limit to 10 results
        
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch category: {str(e)}"})

In [48]:
# Tool: Get meals by area/cuisine
@tool
def get_meals_by_area(area: Annotated[str, "The area/cuisine name (e.g., Canadian, Chinese, Italian)"]) -> str:
    """
    Get a list of meals from a specific area/cuisine (e.g., Canadian, Chinese, Italian).
    Use this when the user wants to explore food from a specific cuisine/region.

    Args:
        area (str): The area/cuisine name (e.g., "Canadian", "Chinese", "Italian").

    Returns:
        str: A JSON string containing list of meals from that area.
    """
    try:
        response = requests.get(
            f"https://www.themealdb.com/api/json/v1/1/filter.php?a={area}"
        )
        response.raise_for_status()
        data = response.json()
        
        if not data.get("meals"):
            return json.dumps({"error": f"No meals found for area '{area}'"})
        
        meals = [{"id": m["idMeal"], "name": m["strMeal"], "thumbnail": m["strMealThumb"]} 
                 for m in data["meals"]]
        return json.dumps(meals[:10], indent=2)  # Limit to 10 results
        
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch area: {str(e)}"})

In [49]:
# Tool: Get all categories
@tool
def get_all_categories() -> str:
    """
    Get a list of all available meal categories.
    Use this to show the user what categories are available.

    Returns:
        str: A JSON string containing all available categories.
    """
    try:
        response = requests.get("https://www.themealdb.com/api/json/v1/1/categories.php")
        response.raise_for_status()
        data = response.json()
        
        if not data.get("categories"):
            return json.dumps({"error": "No categories found"})
        
        categories = [c["strCategory"] for c in data["categories"]]
        return json.dumps({"categories": categories}, indent=2)
        
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch categories: {str(e)}"})

---

## 5. Create the Chat Client

Now let's create the OpenAI chat client that will be used by all our agents.

---

## 6. Create Specialized Sub-Agents

Now we'll create specialized sub-agents that handle specific food-related tasks. Each agent has a specific focus:

- **Meal Details Agent**: Provides detailed information about specific meals
- **Category Explorer Agent**: Helps users explore meals by category
- **Cuisine Explorer Agent**: Helps users explore meals by cuisine/area
- **Random Surprise Agent**: Suggests random meals for indecisive users

In [50]:
# Sub-Agent 1: Meal Details Agent
# This agent specializes in providing detailed information about specific meals

meal_details_agent = Agent(
    name="meal_details_agent",
    client=openai_chat_client,
    instructions="""You are a meal details expert. Your role is to provide comprehensive information about specific meals.
    
    When a user asks about a specific meal:
    1. Use the get_meal_by_name tool to find the meal
    2. Present the information in a friendly, organized manner
    3. Include: name, category, cuisine area, ingredients list, and cooking instructions
    4. If available, mention any YouTube tutorial links
    
    Always be helpful and thorough in your explanations.""",
    tools=[get_meal_by_name]
)

In [51]:
# Sub-Agent 2: Category Explorer Agent
# This agent helps users explore meals by category

category_explorer_agent = Agent(
    name="category_explorer_agent",
    client=openai_chat_client,
    instructions="""You are a category exploration expert. Your role is to help users discover meals by category.
    
    When a user wants to explore meals by category:
    1. First use get_all_categories to show available categories
    2. Ask the user which category interests them
    3. Use get_meals_by_category to list meals in that category
    4. Present the results in a clear, organized list
    
    Be enthusiastic and help guide users to discover new food categories!""",
    tools=[get_all_categories, get_meals_by_category]
)

In [52]:
# Sub-Agent 3: Cuisine Explorer Agent
# This agent helps users explore meals by cuisine/area

cuisine_explorer_agent = Agent(
    name="cuisine_explorer_agent",
    client=openai_chat_client,
    instructions="""You are a cuisine exploration expert. Your role is to help users discover meals from different world cuisines.
    
    When a user wants to explore meals by cuisine:
    1. Show popular cuisine areas (e.g., Italian, Chinese, Mexican, Indian, Japanese, etc.)
    2. Ask the user which cuisine they are interested in
    3. Use get_meals_by_area to list meals from that cuisine
    4. Present the results highlighting the variety of dishes available
    
    Be passionate about world cuisines and help users explore new flavors!""",
    tools=[get_meals_by_area]
)

In [53]:
# Sub-Agent 4: Random Surprise Agent
# This agent suggests random meals for users who cannot decide

random_surprise_agent = Agent(
    name="random_surprise_agent",
    client=openai_chat_client,
    instructions="""You are a food surprise expert. Your role is to help indecisive users by suggesting random meals.
    
    When a user cannot decide what to eat or wants a surprise:
    1. Use get_random_meal to get a random meal suggestion
    2. Present the meal in an exciting, appetizing way
    3. Highlight what makes this dish special
    4. Ask if they would like another suggestion or more details
    
    Be enthusiastic and make the surprise feel exciting!""",
    tools=[get_random_meal]
)

In [54]:
# 7. Create Handoff Rules
# Build the handoff workflow
workflow = (
    HandoffBuilder(
        name="Customer_Food_Support_Handoff",
        participants=[meal_details_agent, category_explorer_agent, cuisine_explorer_agent, random_surprise_agent],
        # Terminate when agent asks a direct question to the user (common pattern for handoff workflows)
        # termination_condition=lambda conversation: len(conversation) > 0 and conversation[-1].text.strip().endswith(""),
        termination_condition=lambda conversation: len(conversation) > 0 and "welcome" in conversation[-1].text.lower(),
    )
    .with_start_agent(meal_details_agent) # Triage receives initial user input
    .with_autonomous_mode()  # Max 3 autonomous turns)
    .build()
)

In [ ]:
# Run the workflow with streaming
async def run_workflow():
    print("=== Running Workflow ===\n")
    
    # Use streaming to get real-time output
    last_response_id = None
    events = workflow.run("What to eat today?", stream=True)
    async for event in events:
        if event.type == "handoff_sent":
            print(f"\nHandoff: {event.data.source} -> {event.data.target}\n")
        elif event.type == "output":
            # AgentResponseUpdate event - streaming text during response
            data = event.data
            if isinstance(data, AgentResponseUpdate):
                if not data.text:
                    continue
                rid = data.response_id
                if rid != last_response_id:
                    if last_response_id is not None:
                        print()  # New line between responses
                    print(f"{data.author_name}:", end=" ", flush=True)
                    last_response_id = rid
                print(data.text, end="", flush=True)

# Run the async function
asyncio.run(run_workflow())

=== Running Workflow ===


Handoff: meal_details_agent -> random_surprise_agent

random_surprise_agent: 🎉 **Surprise!** 🎉  
**Today’s delicious adventure is…** **Pouding Chômeur** – a classic Canadian comfort‑dessert that’ll make your taste buds do a happy dance!  

### Why this treat is a star ✨
- **Sweet & buttery cake base** that’s light, fluffy, and perfectly moist.  
- **Rich, velvety maple sauce** poured over the cake while it bakes, creating a gooey caramel‑like glaze that seeps into every crumb.  
- **Maple‑infused goodness** straight from Canada’s iconic trees – it’s like a warm hug in a slice.  
- **Simple, homestyle ingredients** (butter, sugar, eggs, vanilla, flour, milk, and that glorious maple syrup) that turn an ordinary day into a celebration.  

### Quick peek at the magic
1. **Cream butter & sugar** until airy, then fold in eggs and vanilla.  
2. **Combine dry ingredients** (flour + baking powder) and alternate with milk, creating a silky batter.  
3. **Whip up the ma

---

## 10. Summary

Congratulations! You have successfully implemented the **Handoff Pattern** for the Food Agent.

### What We Built

A **multi-agent system** with the following components:

| Agent | Purpose | Tools |
|-------|---------|-------|
| **Food Agent (Main)** | Coordinates requests and delegates to sub-agents | Handoff tools |
| **Meal Details Agent** | Provides detailed meal information | get_meal_by_name |
| **Category Explorer Agent** | Helps explore meals by category | get_all_categories, get_meals_by_category |
| **Cuisine Explorer Agent** | Helps explore meals by cuisine | get_meals_by_area |
| **Random Surprise Agent** | Suggests random meals | get_random_meal |

### Key Benefits of the Handoff Pattern

- **Specialization**: Each sub-agent excels at its specific task
- **Modularity**: Easy to add or modify sub-agents
- **Scalability**: Can add more specialized agents as needed
- **Flexibility**: Main agent intelligently routes requests

### When to Use Handoff

- When tasks require different expertise
- When you want to decompose complex problems
- When certain agents have specific tools
- When you want to build a team of specialized assistants

---

**Links and Resources**
- [Microsoft Learn - Handoff Pattern](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/handoff?pivots=programming-language-python)
- [Microsoft Agent Framework](https://github.com/microsoft/agent-framework)

**Happy Agent Building!**